# Packer

In this notebook, we will use Packer to create an Ubuntu Server 24.04 LTS image template with support for Cloud-Init.

Packer is a tool that lets you create identical machine images for multiple platforms from a single source template. Packer can create golden images to use in image pipelines. Packer uses a declarative configuration language known as HashiCorp Configuration Language (HCL).

- Packer:
  - https://developer.hashicorp.com/packer
- Packer Install:
  - https://developer.hashicorp.com/packer/install
- Packer Documentation:
  - https://developer.hashicorp.com/packer/docs
- Additonal Links:
  - [Ubuntu Autoinstall](https://canonical-subiquity.readthedocs-hosted.com/en/latest/intro-to-autoinstall.html) 
  - [Debian Autoinstall](https://wiki.debian.org/AutomatedInstallation) 
  - [FAI Project](https://fai-project.org) 
  - [QEMU Packer Plugin](https://developer.hashicorp.com/packer/integrations/hashicorp/qemu)
  - [QEMU Packer Plugin Example](https://developer.hashicorp.com/packer/integrations/hashicorp/qemu/latest/components/builder/qemu)

## Prerequisites

Before we start using Packer, we need to download the ISO file for Ubuntu Server 24.04 LTS. We also need a password, and corresponding password hash, for the default user we intend to use inside the image template we will be creating.

### Download Ubuntu Server 24.04 LTS ISO
- The code below downloads the Ubuntu Server 24.04 LTS ISO file, including a textfile containing the SHA256 checksum using to validate that the ISO hasn't been tampered with.
- The files are placed in the folder `packer/ubuntu-server-24/iso`

**Note:**
- This might take up to 7 minutes.
- Uncomment the other two lines if you also want to download the Ubuntu Desktop 24.04 LTS ISO file (but we will not be using it in our notebooks).

In [1]:
%%bash

wget -O packer/ubuntu-server-24/iso/ubuntu-24.04.4-live-server-amd64.iso https://releases.ubuntu.com/noble/ubuntu-24.04.4-live-server-amd64.iso
wget -O packer/ubuntu-server-24/iso/SHA256SUMS.txt https://releases.ubuntu.com/noble/SHA256SUMS
# wget -O packer/ubuntu-desktop-24/iso/ubuntu-24.04.4-desktop-amd64.iso https://releases.ubuntu.com/noble/ubuntu-24.04.4-desktop-amd64.iso
# wget -O packer/ubuntu-desktop-24/iso/SHA256SUMS.txt https://releases.ubuntu.com/noble/SHA256SUMS

--2026-03-19 19:03:47--  https://releases.ubuntu.com/noble/ubuntu-24.04.4-live-server-amd64.iso
Resolving releases.ubuntu.com (releases.ubuntu.com)... 91.189.91.107, 185.125.190.37, 91.189.91.109, ...
Connecting to releases.ubuntu.com (releases.ubuntu.com)|91.189.91.107|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3405469696 (3,2G) [application/x-iso9660-image]
Saving to: ‘packer/ubuntu-server-24/iso/ubuntu-24.04.4-live-server-amd64.iso’

     0K .......... .......... .......... .......... ..........  0%  169K 5h27m
    50K .......... .......... .......... .......... ..........  0%  257K 4h31m
   100K .......... .......... .......... .......... ..........  0%  475K 3h39m
   150K .......... .......... .......... .......... ..........  0%  517K 3h11m
   200K .......... .......... .......... .......... ..........  0%  700K 2h49m
   250K .......... .......... .......... .......... ..........  0% 1,05M 2h29m
   300K .......... .......... .......... .......... ..

### Password and Password Hash

We will be creating a default user with username `student` inside the Ubuntu Server 24.04 LTS image template, and need to select a password with a corresponding password hash.

- The password we will be using is:
  
  ```bash
  student
  ```
- The SHA-512 hash of that password we will be using is:
  
  ```bash
  $6$Z1nrMvyTXZ6cnOzn$iOgL9h1fIzS6zYslfFibFUTuE7vcZdOjBWDcHcOLq.MChHc.RwPueEhWRNvarY2w3Tife8sfdQoaG4j5fLdFl0
  ```
- The SHA-512 password hash was created using the following code:
  
  ```bash
  openssl passwd -6 student
  ```

## Show Packer version

In [2]:
!packer --version

Packer v1.15.0


## Show Packer help

- The command `packer --help` shows help about terraform commands.
- The command `packer [command] --help` shows additional help about a specific command, e.g. `packer build --help`.

In [3]:
!packer --help

Usage: packer [--version] [--help] <command> [<args>]

Available commands are:
    build           build image(s) from template
    console         creates a console for testing variable interpolation
    fix             fixes templates from old versions of packer
    fmt             Rewrites HCL2 config files to canonical format
    hcl2_upgrade    transform a JSON template into an HCL2 configuration
    init            Install missing plugins or upgrade plugins
    inspect         see components of a template
    plugins         Interact with Packer plugins and catalog
    validate        check that a template is valid
    version         Prints the Packer version



## Packer files

- Packer images are defined in Packer template files with the `.pkr.hcl` and `.pkrvars.hcl` file extensions.
- A folder can contain one or more `.pkr.hcl` and `.pkrvars.hcl` files.
  - Usually, template definitions are spread out among multiple `.pkr.hcl` and `.pkrvars.hcl` files.
- Packer commands are issued within a specific folder containing `.pkr.hcl` and `.pkrvars.hcl` files.
  - All `.pkr.hcl` and `.pkrvars.hcl` files within that folder will be merged into one *file* in memory (RAM) beore the command interprets the code.

#### In this example, the folder we will be issuing Packer commands against contains three `.pkr.hcl` files and one `.pkrvars.hcl` file.

In [4]:
# !dir packer\ubuntu-server-24\*.hcl # use this on Windows
!ls packer/ubuntu-server-24/*.hcl # use this on Linux/Mac

packer/ubuntu-server-24/build.pkr.hcl
packer/ubuntu-server-24/plugins.pkr.hcl
packer/ubuntu-server-24/variables.auto.pkrvars.hcl
packer/ubuntu-server-24/variables.pkr.hcl


## Packer Hashicorp Configuration Language (HCL)

- Each `.hcl` file contains declarations using the Hashicorp Configuration Language (https://developer.hashicorp.com/packer/docs/templates/hcl_templates).
- Hashicorp Configuration Language (HCL) basically consists of `Blocks`, `Arguments` and `Expressions`.

  ```bash
  <BLOCK TYPE> "<BLOCK LABEL>" "<BLOCK LABEL>" {
    # Block body
    <IDENTIFIER> = <EXPRESSION> # Argument
  }
  ```

  - `Blocks`
    - Blocks are containers for other content and usually represent the configuration of some kind of object, like a source or build.
    - Blocks have a `<BLOCK TYPE>`, can have zero or more `<BLOCK LABEL>`s.
    - A Block has a body between `{}` that contains any number of arguments and nested blocks.
    - Most of Packer's features are controlled by top-level blocks in a configuration file.

  - `Arguments`
    - Arguments assign a value to a name, i.e. an `<EXPRESSION>` to an `<IDENTIFIER>`.
    - They appear within a Block body.

  - `Expressions`
    - Expressions represent a value, either literally or by referencing and combining other values.
    - They appear as values for arguments, or within other expressions.
    
  - `Comments`
    - The hash character `#` is used for creating line comments in HCL.

- Here's a concrete example:
  
  ```bash
  # Define an Ubuntu Server image template source
  source "qemu" "ubuntu_server_24" {
    cpus      = 2
    memory    = 4096
    disk_size = "30000M"
    ...
  }
  ```

  - `Block`
    - `source` is the `<BLOCK_TYPE>` which means the Block is defining a **source**.
    - `"qemu"` is the first `<BLOCK_LABEL>` which states what source we are defining (a QEMU source).
    - `"ubuntu_server_24"` is the second `<BLOCK_LABEL>` which is a name we choose to uniquely identify this Block within our packer files.
  - `Arguments`
    - `cpus = 2` is the first argument that assigns the value (expression) `2` to the name (identifier) `cpus`.
    - `memory = 4096` is the second argument that assigns the value (expression) `4096` to the name (identifier) `memory`.
    - `disk_size = "30000M"` is the first argument that assigns the value (expression) `"30000M"` to the name (identifier) `disk_size`.

## Let's view the contents of the file  `plugins.pkr.hcl`

- This file configures **packer pugins**.
  - A **plugin** adds functionality for defining **sources** for building images for a specific system.
  - In this case, only one plugin `qemu` is configured, providing functionality for defining **QEMU** sources.

- Hashicorp Configuration Language (HCL)
  - At the root level, the file contains one block with block type `packer`.
  - The `packer` block is used to configure the `packer` CLI tool, and contains:
    - A nested block `required_plugins` used to specify Packer plugins to use, which contains one argument:
      - `qemu = { version = ">= 1.1.0"     source = "github.com/hashicorp/qemu" }`.
        - This states that at least version `1.1.0` of the `hashicorp/qemu` plugin is required.
        - It will be downloaded from the Hashicorp plugin registry (https://releases.hashicorp.com) as described here: https://developer.hashicorp.com/packer/docs/plugins.

In [5]:
# !type packer\ubuntu-server-24\plugins.pkr.hcl # use this on Windows
!cat packer/ubuntu-server-24/plugins.pkr.hcl # use this on Linux/Mac

packer {
  required_plugins {
    qemu = {
      version = ">= 1.1.0"
      source  = "github.com/hashicorp/qemu"
    }
  }
}

## Initialize a Packer Project

- The command `packer init .` initializes a Packer project (i.e. for the current folder).
  - It will download the configured plugin `qemu` (in `plugins.pkr.hcl`).

In [6]:
%%bash

cd packer/ubuntu-server-24
packer init .

Installed plugin github.com/hashicorp/qemu v1.1.4 in "/home/student/.config/packer/plugins/github.com/hashicorp/qemu/packer-plugin-qemu_v1.1.4_x5.0_linux_amd64"


## Let's view the contents of the file `variables.pkr.hcl`

- This file defines Packer variables, where a variable is defined as a name-value pair.
  - Packer variables can be referenced in other `.hcl` files in the same Packer project (folder).
- In this case seven variables are defined.
  - The variable with name `iso_url` and value `iso/ubuntu-24.04.4-live-server-amd64.iso`.
    - This variable is used to set the location of the Ubuntu Server ISO file from which an image template will be created.
    - Notice how a variables value is set using `default = "<value>"`, where `<value>` is the variable's value.
    - Also notice how a variable can be described using `description = "<text>"`, where `<text>` is the variable's description.
    - A value can also be provided via an environment variable.
  - The variable with name `iso_checksum` contains a checksum as its value, which is used to validate the downloaded Ubuntu Server ISO file hasn't been tampered with.
  - The variable with name `vm_hostname` sets the hostname that will be used inside the image template.
  - The variable with name `vm_realname` sets the default user's name that will be used inside the image template (this name shows up on login screens).
  - The variable with name `vm_username` sets the default user's username that will be used inside the image template.
  - The variable with name `vm_password` sets the default user's password that will be used inside the image template.
    - Notice that this variable doesn't have a value defined, since it contains a password (its value is usually set in another `.hcl` file or as an environment variable).
    - Also notice the setting `sensitive = true` which tells Packer that this variable is sensitive (since it contains a password).
  - The variable with name `vm_password_hash` contains a SHA-512 hash of the default user's password.
    - Notice that this variable also lacks a value, and has `sensitive = true`.

In [7]:
#!type packer\ubuntu-server-24\variables.pkr.hcl # use this on Windows
!cat packer/ubuntu-server-24/variables.pkr.hcl # use this on Linux/Mac

variable "iso_url" {
  type        = string
  description = "ISO to create a template from"
  # default     = "https://releases.ubuntu.com/noble/ubuntu-24.04.4-live-server-amd64.iso"
  default = "iso/ubuntu-24.04.4-live-server-amd64.iso"
}

variable "iso_checksum" {
  type        = string
  description = "SHA256 checksum for the ISO (format: sha256:<hex>)"
  # Checksum found here: https://releases.ubuntu.com/noble/SHA256SUMS
  default = "sha256:e907d92eeec9df64163a7e454cbc8d7755e8ddc7ed42f99dbc80c40f1a138433"
}

variable "vm_hostname" {
  type        = string
  description = "The VM template's hostname"
  default     = "ubuntu-server-24-template"
}

variable "vm_realname" {
  type        = string
  description = "The default user's name inside the VM"
  default     = "Student"
}

variable "vm_username" {
  type        = string
  description = "The default user's username inside the VM"
  default     = "student"
}

variable "vm_password" {
  type        = string
  description = "The def

## Let's view the contents of the file `variables.auto.pkrvars.hcl`

- This file assigns values to variables defined in another `.pkr.hcl` file.
  - Notice the file extension `.auto.pkrvars.hcl`.
- In this case, we are assigning values to the two variables `vm_hash` and `vm_password_hash` defined in `variables.pkr.hcl`.

In [8]:
#!type packer\ubuntu-server-24\variables.auto.pkrvars.hcl # use this on Windows
!cat packer/ubuntu-server-24/variables.auto.pkrvars.hcl # use this on Linux/Mac

vm_password      = "student"
vm_password_hash = "$6$Z1nrMvyTXZ6cnOzn$iOgL9h1fIzS6zYslfFibFUTuE7vcZdOjBWDcHcOLq.MChHc.RwPueEhWRNvarY2w3Tife8sfdQoaG4j5fLdFl0"

## Let's view the contents of the file `build.pkr.hcl`

- This file defines a QEMU `source`.
  - A `source` is used to define an image template for a specific virtualization system, `qemu` in this case (other possible systems are `amazon`, `ansible`, `azure`, `docker`, `googlecompute`, `kubevirt`, `vagrant`, and `virtualbox`).
- A source is defined using:
  - The **Block Type** `source`.
  - Followed by a **Block Label** containing:
    - The name of the plugin (in this case `qemu`).
    - Followed by a space.
    - Then an arbitrary name you want to use to uniquely refer to this specific QEMU `source` instance (it can be any name you like).
     - This is used to identify it in this and other Packer files.
  - Everything within brackets (`{ }`) is the **Block Body** which contains **Arguments** for configuring the source (image template).
   - Exactly what goes within this block is specific to the QEMU plugin.
- Near the end of the file, there is another block of type `build`.
  - This block is used to actually build the image temple.
  - The argument `sources = ["source.qemu.ubuntu_server_24"]` specifies that we want to use the `source` we defined above to build an image template.
  - Within this block, there is another block of type `provisioner` with an attribute of `shell`.
    - Provisioners are used to configure an image before it is converted into a template (see https://developer.hashicorp.com/packer/docs/provisioners).
    - A `shell` provisioner executes commands in the image's shell.
    - In the sample file, this is used to set up the image for `cloud-init` (see https://cloud-init.io).

In [9]:
#!type packer\ubuntu-server-24\build.pkr.hcl # use this on Windows
!cat packer/ubuntu-server-24/build.pkr.hcl # use this on Linux/Mac

source "qemu" "ubuntu_server_24" {
  accelerator      = "kvm"
  cpus             = 2
  memory           = 4096
  disk_size        = "30000M"
  format           = "qcow2"
  output_directory = "template"
  vm_name          = "ubuntu-server-24-template.qcow2"
  headless         = false

  iso_url      = var.iso_url
  iso_checksum = var.iso_checksum

  # http_directory = "http"
  http_content = {
    "/user-data" = templatefile("${path.root}/http/user-data.pkrtpl.hcl", {
      realname      = var.vm_realname
      username      = var.vm_username
      password      = var.vm_password
      password_hash = var.vm_password_hash
      hostname      = var.vm_hostname
    })

    "/meta-data" = templatefile("${path.root}/http/meta-data.pkrtpl.hcl", {
      instance_id    = var.vm_hostname
      local_hostname = var.vm_hostname
    })
  }

  # Using UEFI (to use BIOS, you would remove these 4 lines).
  firmware          = "efi"
  efi_boot          = true
  efi_firmware_code = "/usr/share/OVMF/OVM

## Packer Fmt

- The command `packer fmt .` formats `.hcl` files in the current folder (e.g. correct indentation, etc.).

In [10]:
%%bash

cd packer/ubuntu-server-24
packer fmt .

## Packer Validate

- The command `packer validate .` validates `.hcl` files in the current folder for correct syntax.

In [11]:
%%bash

cd packer/ubuntu-server-24
packer validate .

The configuration is valid.


## Autoinstall

Before we actually build (create) the image template, let's look at the two files in the folder `packer/ubuntu-server-24/http`.
- `meta-data.pkrtpl.hcl` which is used to set the `instance-id` and `local-hostname` for the image template.
- `user-data.pkrtpl.hcl` which is used to autoinstall Ubuntu Server 24.04 LTS and to perform some initial configuration of the image template.

Notice the file extension `.pkrtpl.hcl` which makes these files Packer Template files, i.e. they can contain variables that we can set from `build.pkr.hcl`.

In [12]:
#!type packer\ubuntu-server-24\http\meta-data.pkrtpl.hcl # use this on Windows
!cat packer/ubuntu-server-24/http/meta-data.pkrtpl.hcl # use this on Linux/Mac

instance-id: ${instance_id}
local-hostname: ${local_hostname}


In [13]:
#!type packer\ubuntu-server-24\http\user-data.pkrtpl.hcl # use this on Windows
!cat packer/ubuntu-server-24/http/user-data.pkrtpl.hcl # use this on Linux/Mac

#cloud-config
autoinstall:
  version: 1

  locale: en_US.UTF-8
  timezone: Europe/Stockholm

  keyboard:
    layout: se
    variant: ""

  storage:
    layout:
      name: direct

  updates: all

  packages:
    - qemu-guest-agent
    - openssh-server

  ssh:
    install-server: true
    allow-pw: true

  identity:
    realname: "${realname}"
    username: "${username}"
    password: "${password_hash}"
    hostname: "${hostname}"

  late-commands:
    - curtin in-target --target=/target -- bash -c 'echo "${username}:${password}" | chpasswd'
    - curtin in-target --target=/target -- usermod -aG sudo ${username}
    - curtin in-target --target=/target -- bash -c 'echo "${username} ALL=(ALL) NOPASSWD:ALL" > /etc/sudoers.d/99-${username}'
    - curtin in-target --target=/target -- chmod 0440 /etc/sudoers.d/99-${username}
    - curtin in-target --target=/target -- systemctl enable qemu-guest-agent
    - curtin in-target --target=/target -- systemctl enable ssh


## Packer Build

Now we are ready to build (create) the Ubuntu Server 24.04 LTS image template, where the resulting file `ubuntu-server-24-template.qcow2` will be placed in the folder `packer/ubuntu-server-24/template`.

- The command `packer build .` builds the image template using all `.hcl` files in the current directory.

**Note**
- During the build process, a window will open (since we have `headless = false` in `build.pkr.hcl`) so you can see the build process.
- Don't touch the window or close it.
- The build process might take up to 11 minutes.

In [14]:
%%bash

cd packer/ubuntu-server-24
packer build .

qemu.ubuntu_server_24: output will be in this color.

==> qemu.ubuntu_server_24: Retrieving ISO
==> qemu.ubuntu_server_24: Trying iso/ubuntu-24.04.4-live-server-amd64.iso
==> qemu.ubuntu_server_24: Trying iso/ubuntu-24.04.4-live-server-amd64.iso?checksum=sha256%3Ae907d92eeec9df64163a7e454cbc8d7755e8ddc7ed42f99dbc80c40f1a138433
==> qemu.ubuntu_server_24: iso/ubuntu-24.04.4-live-server-amd64.iso?checksum=sha256%3Ae907d92eeec9df64163a7e454cbc8d7755e8ddc7ed42f99dbc80c40f1a138433 => /home/<sensitive>/projects/devops/workshop5/01_Infrastructure_as_Code/packer/ubuntu-server-24/iso/ubuntu-24.04.4-live-server-amd64.iso
==> qemu.ubuntu_server_24: Starting HTTP server on port 8827
==> qemu.ubuntu_server_24: Found port for communicator (SSH, WinRM, etc): 3368.
==> qemu.ubuntu_server_24: Creating temporary RSA SSH key for instance...
==> qemu.ubuntu_server_24: Looking for available port between 5900 and 6000 on 127.0.0.1
==> qemu.ubuntu_server_24: Starting VM, booting from CD-ROM
==> qemu.ubuntu_se